# Engenharia de Features e Pré-processamento Blindado contra Vazamento de Dados

Este notebook corresponde à **Etapa 5** do nosso Roteiro de Desenvolvimento do **Tech Challenge (Fase 1)**. 

### 🎯 Objetivo desta Etapa:
O principal foco do pré-processamento em Ciência de Dados é a **blindagem contra o Vazamento de Dados (*Data Leakage*)**. O vazamento ocorre quando informações do conjunto de teste (ou dados futuros) "vazam" para o conjunto de treinamento através de cálculos de estatísticas globais (como a média e desvio padrão para normalização ou preenchimento de nulos).

Para evitar esse erro clássico, utilizaremos a biblioteca **Scikit-Learn** para construir pipelines modulares de transformação:
1. **`ColumnTransformer`**: Permite aplicar transformações independentes e paralelas em diferentes tipos de colunas (numéricas vs. categóricas).
2. **`StandardScaler`**: Padronização de variáveis numéricas de forma isolada.
3. **`OneHotEncoder`**: Codificação robusta de variáveis categóricas com tratamento para categorias desconhecidas (`handle_unknown='ignore'`).
4. **`SimpleImputer`**: Imputação segura para evitar falhas com eventuais dados faltantes.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configuração de logs simples
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 📂 1. Carregando as Configurações e Dados Processados

Vamos importar as variáveis de configuração que definimos na **Etapa 2** (`src/config.py`) para garantir que as exclusões de colunas sensíveis e futuras ocorram exatamente como planejado.

In [2]:
# Ajuste do caminho relativo para rodar a partir do repositório local
BASE_DIR = Path(os.getcwd()).resolve()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "processed_nps_data.csv"

# Fallback caso esteja rodando num ambiente temporário
if not PROCESSED_DATA_PATH.exists():
    PROCESSED_DATA_PATH = Path("processed_nps_data.csv")

print(f"Carregando dados processados de: {PROCESSED_DATA_PATH.resolve()}")
df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Dataset carregado! Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
df.head()

Carregando dados processados de: /home/userdev/dev/projetos/POSTECH_AI_SCIENTIST/TECH_CHALLENGE_ECOMMERCE/data/processed/processed_nps_data.csv
Dataset carregado! Linhas: 2500 | Colunas: 23


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,...,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score,is_detractor,delay_group,contacts_group,support_group
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,...,0.0,4,6.9,0,3,6.5,0,Atraso Baixo (1-2 dias),Sem Contato,Médio (3-5 dias)
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,...,0.0,10,2.4,0,3,0.0,1,Atraso Médio (3-4 dias),Sem Contato,Crítico (9+ dias)
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,...,3.5,5,4.8,0,7,1.5,1,Atraso Baixo (1-2 dias),Contato Médio (3-4 contatos),Médio (3-5 dias)
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,...,1.0,11,5.9,0,4,0.3,1,Atraso Baixo (1-2 dias),Contato Baixo (1-2 contatos),Crítico (9+ dias)
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,...,1.0,0,6.1,0,3,7.9,0,Atraso Baixo (1-2 dias),Contato Baixo (1-2 contatos),Rápido (0-2 dias)


## 🔒 2. Divisão de Features (X) e Target (y)

Para evitar qualquer vazamento de dados, excluiremos colunas de IDs (`customer_id`, `order_id`), a nota contínua de origem (`nps_score`), e as informações que acontecem após a entrega ou em momentos incertos (`repeat_purchase_30d` e `csat_internal_score`).

In [3]:
# Exclusões estritas baseadas no config.py
EXCLUDED_COLS = [
    "customer_id",
    "order_id",
    "nps_score",
    "repeat_purchase_30d",
    "csat_internal_score"
]

# Também removemos colunas de faixas categóricas auxiliares que criamos apenas para a AED do Passo 4
AUX_COLS = ["delay_group", "contacts_group", "support_group"]

drop_cols = EXCLUDED_COLS + AUX_COLS + ["is_detractor"]
available_drops = [col for col in drop_cols if col in df.columns]

# Divisão X e y
X = df.drop(columns=available_drops)
y = df["is_detractor"]

print("=== SHAPES ORIGINAIS ===")
print(f"X (Features): {X.shape}")
print(f"y (Target): {y.shape}")
print(f"Proporção de Detratores: {y.mean() * 100:.2f}%")

=== SHAPES ORIGINAIS ===
X (Features): (2500, 14)
y (Target): (2500,)
Proporção de Detratores: 74.04%


## 🛠️ 3. Separação de Tipos de Colunas para Pré-processamento

Precisamos separar as features em numéricas (que serão escalonadas) e categóricas (que serão convertidas em variáveis binárias One-Hot).

In [4]:
num_features = []
cat_features = []

for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        num_features.append(col)
    else:
        cat_features.append(col)

print(f"-> Features Numéricas ({len(num_features)}):\n{num_features}\n")
print(f"-> Features Categóricas ({len(cat_features)}):\n{cat_features}")

-> Features Numéricas (13):
['customer_age', 'customer_tenure_months', 'order_value', 'items_quantity', 'discount_value', 'payment_installments', 'delivery_time_days', 'delivery_delay_days', 'freight_value', 'delivery_attempts', 'customer_service_contacts', 'resolution_time_days', 'complaints_count']

-> Features Categóricas (1):
['customer_region']


## ⚙️ 4. Construindo o ColumnTransformer do Scikit-Learn

Agora criamos o pipeline de pré-processamento. Este pipeline unificado se integrará perfeitamente aos modelos preditivos nas próximas etapas de treinamento.

In [ ]:
# 1. Sub-pipeline Numérico
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # Imputação segura pela mediana
    ("scaler", StandardScaler())                 # Padronização (Z-score)
])

# 2. Sub-pipeline Categórico
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)) # Evita quebra com dados desconhecidos
])

# 3. Unificador Global (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features)
    ],
    remainder="drop" # Descarta qualquer outra coluna que passe batida
)

print("✓ ColumnTransformer instanciado com sucesso!")

✓ ColumnTransformer instanciado com sucesso!


## 🧪 5. Executando um fit_transform de Validação

Para certificar que nosso pipeline está pronto para entrar em produção, vamos simular a transformação sobre o nosso X e inspecionar a base resultante.

In [6]:
# Aplicando o pré-processador
X_trans = preprocessor.fit_transform(X)

print("=== COMPARAÇÃO DE DIMENSÕES ===")
print(f"Shape original de X: {X.shape}")
print(f"Shape após transformações: {X_trans.shape}")

# Obtendo o nome das colunas transformadas após o OneHotEncoder para fins visuais
encoded_cats = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
all_feature_names = num_features + list(encoded_cats)

print(f"\n✓ Total de colunas geradas no pipeline: {len(all_feature_names)}")
print("Nomes das colunas pós-transformação:")
for i, col_name in enumerate(all_feature_names):
    print(f" - [{i}]: {col_name}")

=== COMPARAÇÃO DE DIMENSÕES ===
Shape original de X: (2500, 14)
Shape após transformações: (2500, 18)

✓ Total de colunas geradas no pipeline: 18
Nomes das colunas pós-transformação:
 - [0]: customer_age
 - [1]: customer_tenure_months
 - [2]: order_value
 - [3]: items_quantity
 - [4]: discount_value
 - [5]: payment_installments
 - [6]: delivery_time_days
 - [7]: delivery_delay_days
 - [8]: freight_value
 - [9]: delivery_attempts
 - [10]: customer_service_contacts
 - [11]: resolution_time_days
 - [12]: complaints_count
 - [13]: customer_region_Centro-Oeste
 - [14]: customer_region_Nordeste
 - [15]: customer_region_Norte
 - [16]: customer_region_Sudeste
 - [17]: customer_region_Sul


## 📝 6. Visualizando a Matriz Resultante

Vamos converter a matriz NumPy transformada em um DataFrame pandas apenas para inspecionar os primeiros registros e comprovar que os dados estão normalizados de forma correta e sem nulos.

In [7]:
df_trans = pd.DataFrame(X_trans, columns=all_feature_names)
print("Estatísticas resumidas após padronização (Média ~0, Desvio Padrão ~1):")
display(df_trans.describe().round(3).head(3)) # Médias e desvios padronizados

print("\nPrimeiros 3 registros pós-processamento:")
df_trans.head(3)

Estatísticas resumidas após padronização (Média ~0, Desvio Padrão ~1):


,customer_age,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count,customer_region_Centro-Oeste,customer_region_Nordeste,customer_region_Norte,customer_region_Sudeste,customer_region_Sul
count,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0,2500.000,2500.000,2500.000,2500.000,2500.000
mean,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.187,0.194,0.202,0.208,0.208
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.390,0.396,0.402,0.406,0.406



Primeiros 3 registros pós-processamento:


,customer_age,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count,customer_region_Centro-Oeste,customer_region_Nordeste,customer_region_Norte,customer_region_Sudeste,customer_region_Sul
0,1.316986,-1.372785,-1.016621,0.313694,0.328695,-0.634356,-1.597493,-0.125370,1.433947,1.219624,-1.330419,-0.429698,-0.652376,0.0,1.0,0.0,0.0,0.0
1,-1.571730,-1.749904,0.085223,-0.871847,-0.692532,1.264913,-0.536388,1.273851,-0.827174,1.219624,-1.330419,1.305755,-0.652376,0.0,0.0,0.0,0.0,1.0
2,0.174935,1.441107,0.251283,0.906465,0.447450,-0.001266,-0.536388,-0.824981,0.229672,-1.233360,1.855450,-0.140456,1.651211,0.0,1.0,0.0,0.0,0.0
